In [19]:
import yaml
import json
import os

In [20]:
resources_dir = os.path.join("..", "resources")
data_dir = os.path.join("..", "data")

In [21]:
bidsmap = None
with open(os.path.join(resources_dir, "prisma_test.yaml"), "r") as stream:
    try:
        bidsmap = yaml.safe_load(stream)
    except yaml.YAMLError as error:
        print(error)
bidsmap

{'__bids__': '1.10.0',
 'MRI': {'jsonNIFTI': {'__ignore__': [{'provenance': 'prepared/sub-A085/ses-01/MRI/001-localizer/AA_085_1_localizer_i00001.nii.gz',
     'checked': True,
     'model': '__unknown__',
     'suffix': '',
     'attributes': {'ProtocolName': 'localizer'},
     'bids': [],
     'json': []},
    {'provenance': 'prepared/sub-C225/ses-01/MRI/001-AAHead_Scout/COV225_1_AAHead_Scout.nii.gz',
     'checked': True,
     'model': '__unknown__',
     'suffix': '',
     'attributes': {'ProtocolName': 'AAHead_Scout'},
     'bids': [],
     'json': []},
    {'provenance': 'prepared/sub-A085/ses-01/MRI/005-Field_mapping/AA_085_5_Field_mapping_e2_ph.nii.gz',
     'checked': True,
     'model': '__unknown__',
     'suffix': '',
     'attributes': {'ProtocolName': 'Field_mapping'},
     'bids': [],
     'json': []},
    {'provenance': 'prepared/sub-A085/ses-01/MRI/006-MB_Resting_SBRef/AA_085_6_MB_Resting.nii.gz',
     'checked': True,
     'model': '__unknown__',
     'suffix': '',
  

In [25]:
prisma_config = {"descriptions": []}
for datatype in bidsmap["MRI"]["jsonNIFTI"]:
    if datatype != "__ignore__":
        for bidsmap_entry in bidsmap["MRI"]["jsonNIFTI"][datatype]:
            if "ManufacturersModelName" in bidsmap_entry["attributes"]:
                continue
            new_atts = {}
            for att in bidsmap_entry["attributes"]:
                new_key = att.strip("<>")
                if type(bidsmap_entry["attributes"][att]) == str:
                    new_atts[new_key] = bidsmap_entry["attributes"][att].replace("_", "?")
                else:
                    new_atts[new_key] = bidsmap_entry["attributes"][att]

            bidsmap_entry["attributes"] = new_atts

            config_entry = {
                "datatype": datatype,
                "suffix": bidsmap_entry["suffix"],
                "criteria": bidsmap_entry["attributes"],
                "sidecar_changes": {
                    "InstitutionName": "Rutgers University Brain Imaging Center (RUBIC)",
                    "InstitutionAddress": "Univeristy Ave. 197,Newark,North East,US,07102-1814", 
                    "InstitutionalDepartmentName": "Center for Molecular and Behavioral Neuroscience- Aging and Brain Health Alliance"}
            }
            prisma_config["descriptions"].append(config_entry)
prisma_config

{'descriptions': [{'datatype': 'func',
   'suffix': 'bold',
   'criteria': {'ProtocolName': 'MB?fMRI?resting?state', 'ImageType/2': 'M'},
   'sidecar_changes': {'InstitutionName': 'Rutgers University Brain Imaging Center (RUBIC)',
    'InstitutionAddress': 'Univeristy Ave. 197,Newark,North East,US,07102-1814',
    'InstitutionalDepartmentName': 'Center for Molecular and Behavioral Neuroscience- Aging and Brain Health Alliance'}},
  {'datatype': 'anat',
   'suffix': 'T1w',
   'criteria': {'ProtocolName': 'MPRage'},
   'sidecar_changes': {'InstitutionName': 'Rutgers University Brain Imaging Center (RUBIC)',
    'InstitutionAddress': 'Univeristy Ave. 197,Newark,North East,US,07102-1814',
    'InstitutionalDepartmentName': 'Center for Molecular and Behavioral Neuroscience- Aging and Brain Health Alliance'}},
  {'datatype': 'anat',
   'suffix': 'FLAIR',
   'criteria': {'ProtocolName': ['Sagittal 3D FLAIR', 'Sagittal_3D_FLAIR']},
   'sidecar_changes': {'InstitutionName': 'Rutgers University 

In [23]:
prisma_config["descriptions"] = [desc for desc in prisma_config["descriptions"] if "ManufacturersModelName" not in desc["criteria"]]

for desc in prisma_config["descriptions"]:
    if("ManufacturersModelName" in desc["criteria"]):
        print(desc)

In [24]:
with open(os.path.join(resources_dir, "prisma_config.json"), "w") as config_file:
    json.dump(prisma_config, indent=4, fp=config_file)